# MODUL PRAKTIKUM BIG DATA
## Pertemuan 5 — PySpark DataFrame Lanjutan: Join, Window Function & Spark SQL

| | |
|---|---|
| **Mata Kuliah** | Praktikum Big Data |
| **Program Studi** | Teknologi Informasi — Universitas Tidar |
| **Pertemuan** | 5 |
| **Topik** | Join antar-DataFrame, Window Function, Spark SQL |
| **Estimasi Waktu** | 3 x 50 menit |
| **Prasyarat** | Modul Pertemuan 1-4 selesai (Hadoop + HDFS aktif, Spark 3.5.9 + PySpark 3.5.9 sudah terinstall) |

---

pastikan Hadoop aktif (`start-dfs.sh` dan `start-yarn.sh`), lalu buka modul ini di Jupyter.

> **Konsistensi versi:** Tidak ada perubahan versi pada pertemuan ini — tetap **Spark/PySpark 3.5.9**, **Python 3.11** (environment `bigdata`), **Hadoop 3.4.3**. Modul ini murni memperdalam kemampuan PySpark yang sudah terinstall.


---
## Recap Pertemuan Sebelumnya

- [ ] Hadoop aktif (`jps` menampilkan 5 proses)
- [ ] `conda activate bigdata` berhasil
- [ ] Sudah memahami: `SparkSession`, `select`, `filter`, `groupBy`, `agg` dari Pertemuan 4

## Tujuan Pembelajaran

Setelah menyelesaikan Pertemuan 5, mahasiswa mampu:
1. Menggabungkan (*join*) dua atau lebih Spark DataFrame menggunakan berbagai jenis join.
2. Menerapkan *window function* untuk melakukan perankingan data dalam kelompok tertentu.
3. Menulis dan menjalankan kueri SQL langsung di atas Spark DataFrame menggunakan Spark SQL.
4. Memilih pendekatan (DataFrame API vs Spark SQL) yang paling sesuai untuk suatu kebutuhan analisis.

---

## Persiapan: Membuat SparkSession & Dataset

Seperti biasa, jalankan cell ini terlebih dahulu di setiap sesi Jupyter baru.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, avg, count, rank, row_number, when
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("Pertemuan5-JoinWindowSQL") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession siap. Versi Spark:", spark.version)

26/09/19 10:38:37 WARN Utils: Your hostname, HP resolves to a loopback address: 127.0.1.1; using 192.168.0.106 instead (on interface wlo1)
26/09/19 10:38:37 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/19 10:38:38 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession siap. Versi Spark: 3.5.9


Pada pertemuan ini kita akan bekerja dengan **dua tabel** sekaligus — mensimulasikan skenario dunia nyata di mana data tersebar di beberapa sumber dan perlu digabungkan (persis seperti *join* pada SQL/database relasional):

1. **`df_transaksi`** — data transaksi (seperti pertemuan-pertemuan sebelumnya)
2. **`df_produk`** — tabel referensi/master berisi target penjualan bulanan & nama manager per kategori produk

In [3]:
import numpy as np
import pandas as pd

np.random.seed(7)
kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga"]

# Tabel referensi/master: target & manager per kategori (data ini relatif statis, jarang berubah)
data_produk = {
    "kategori": kategori_list,
    "target_bulanan": [50000000, 40000000, 30000000, 25000000, 20000000],
    "manager": ["Andi", "Budi", "Citra", "Dewi", "Eka"],
}
df_produk = spark.createDataFrame(pd.DataFrame(data_produk))

# Tabel transaksi: data yang terus bertambah setiap hari
n = 300
data_transaksi = {
    "order_id": [f"O{i}" for i in range(n)],
    "kategori": np.random.choice(kategori_list, size=n),
    "kota": np.random.choice(["Magelang", "Semarang", "Solo"], size=n),
    "pendapatan": np.random.randint(50000, 500000, size=n),
}
df_transaksi = spark.createDataFrame(pd.DataFrame(data_transaksi))

print("df_produk:")
df_produk.show()
print("df_transaksi (5 baris pertama dari total", df_transaksi.count(), "baris):")
df_transaksi.show(5)

df_produk:


+--------------------+--------------+-------+
|            kategori|target_bulanan|manager|
+--------------------+--------------+-------+
|          Elektronik|      50000000|   Andi|
|             Fashion|      40000000|   Budi|
|   Makanan & Minuman|      30000000|  Citra|
|Kesehatan & Kecan...|      25000000|   Dewi|
|        Rumah Tangga|      20000000|    Eka|
+--------------------+--------------+-------+

df_transaksi (5 baris pertama dari total 300 baris):
+--------+--------------------+--------+----------+
|order_id|            kategori|    kota|pendapatan|
+--------+--------------------+--------+----------+
|      O0|        Rumah Tangga|Magelang|    488643|
|      O1|             Fashion|Magelang|    401943|
|      O2|Kesehatan & Kecan...|    Solo|    452308|
|      O3|Kesehatan & Kecan...|Semarang|    421741|
|      O4|        Rumah Tangga|Semarang|    185244|
+--------+--------------------+--------+----------+
only showing top 5 rows



---
## 5.1 Menggabungkan DataFrame dengan `join()`

Sintaks umum: **`df1.join(df2, on="kolom_kunci", how="jenis_join")`**

| Jenis Join | Hasil |
|---|---|
| `inner` (default) | Hanya baris yang memiliki kecocokan di **kedua** tabel |
| `left` | Seluruh baris dari tabel kiri, dilengkapi data tabel kanan jika cocok (kosong/`null` jika tidak) |
| `right` | Kebalikan dari `left` |
| `outer` (atau `full`) | Seluruh baris dari **kedua** tabel, cocok maupun tidak |

### 5.1.1 Left Join — Melihat Detail Transaksi Beserta Info Manager

In [4]:
df_gabung = df_transaksi.join(df_produk, on="kategori", how="left")
df_gabung.show(5)

+--------------------+--------+--------+----------+--------------+-------+
|            kategori|order_id|    kota|pendapatan|target_bulanan|manager|
+--------------------+--------+--------+----------+--------------+-------+
|Kesehatan & Kecan...|      O2|    Solo|    452308|      25000000|   Dewi|
|Kesehatan & Kecan...|      O3|Semarang|    421741|      25000000|   Dewi|
|             Fashion|      O1|Magelang|    401943|      40000000|   Budi|
|             Fashion|      O5|Semarang|    155828|      40000000|   Budi|
|        Rumah Tangga|      O0|Magelang|    488643|      20000000|    Eka|
+--------------------+--------+--------+----------+--------------+-------+
only showing top 5 rows



### 5.1.2 Inner Join Setelah Agregasi — Kasus yang Lebih Realistis

Pola yang jauh lebih umum dipakai di dunia kerja: **meringkas data transaksi terlebih dahulu**, baru digabungkan dengan tabel referensi untuk dibandingkan terhadap target.

In [5]:
# Langkah 1: Meringkas total pendapatan per kategori
ringkasan = df_transaksi.groupBy("kategori").agg(
    spark_sum("pendapatan").alias("total_pendapatan")
)

# Langkah 2: Join dengan tabel target untuk menghitung pencapaian
hasil = ringkasan.join(df_produk, on="kategori", how="inner")
hasil = hasil.withColumn(
    "pencapaian_persen",
    (col("total_pendapatan") / col("target_bulanan") * 100)
)

hasil.orderBy(col("pencapaian_persen").desc()).show()

+--------------------+----------------+--------------+-------+------------------+
|            kategori|total_pendapatan|target_bulanan|manager| pencapaian_persen|
+--------------------+----------------+--------------+-------+------------------+
|        Rumah Tangga|        17243718|      20000000|    Eka| 86.21858999999999|
|Kesehatan & Kecan...|        17607693|      25000000|   Dewi| 70.43077199999999|
|   Makanan & Minuman|        13338394|      30000000|  Citra| 44.46131333333334|
|             Fashion|        16588742|      40000000|   Budi|41.471855000000005|
|          Elektronik|        16300808|      50000000|   Andi|         32.601616|
+--------------------+----------------+--------------+-------+------------------+



> Perhatikan pola kerja di atas: **agregasi dulu, baru join** — jauh lebih efisien dibanding join dulu baru agregasi, karena jumlah baris yang di-join menjadi jauh lebih sedikit (5 baris kategori, bukan 300 baris transaksi mentah).

---

## 5.2 Window Function

*Window function* memungkinkan kita melakukan perhitungan **dalam kelompok (partisi) tertentu** tanpa meringkas/menghilangkan baris aslinya — berbeda dengan `groupBy` yang meringkas data menjadi lebih sedikit baris. Kegunaan paling umum: **perankingan** dalam suatu kelompok.

**Kasus:** Kita ingin tahu, **untuk setiap kategori**, transaksi mana yang pendapatannya paling tinggi (top-2 misalnya) — tanpa kehilangan detail baris transaksinya.

In [6]:
# Mendefinisikan "window": kelompokkan berdasarkan kategori, urutkan dari pendapatan tertinggi
window_spec = Window.partitionBy("kategori").orderBy(col("pendapatan").desc())

# Menambahkan kolom peringkat SETIAP transaksi di dalam kategorinya masing-masing
df_ranked = df_transaksi.withColumn("rank_dalam_kategori", rank().over(window_spec))

# Menampilkan hanya 2 transaksi teratas (rank 1 dan 2) di setiap kategori
df_ranked.filter(col("rank_dalam_kategori") <= 2) \
    .orderBy("kategori", "rank_dalam_kategori") \
    .show(10)

+--------+--------------------+--------+----------+-------------------+
|order_id|            kategori|    kota|pendapatan|rank_dalam_kategori|
+--------+--------------------+--------+----------+-------------------+
|    O197|          Elektronik|Magelang|    495470|                  1|
|      O6|          Elektronik|Semarang|    491344|                  2|
|     O96|             Fashion|Magelang|    498770|                  1|
|     O42|             Fashion|Semarang|    492607|                  2|
|     O50|Kesehatan & Kecan...|Magelang|    497864|                  1|
|     O72|Kesehatan & Kecan...|    Solo|    478508|                  2|
|    O215|   Makanan & Minuman|Magelang|    493687|                  1|
|    O190|   Makanan & Minuman|Semarang|    470049|                  2|
|    O286|        Rumah Tangga|    Solo|    497826|                  1|
|    O147|        Rumah Tangga|Semarang|    497370|                  2|
+--------+--------------------+--------+----------+-------------

> **`rank()` vs `row_number()`:** `rank()` memberi peringkat sama jika nilainya persis sama (dan melompati angka berikutnya, mis. 1,2,2,4), sedangkan `row_number()` **selalu** memberi nomor urut unik meski nilainya sama (1,2,3,4). Pilih sesuai kebutuhan — gunakan `row_number()` jika anda butuh memastikan jumlah baris yang tepat (misalnya benar-benar hanya top-2, bukan top-2-atau-lebih-jika-seri).


---
## 5.3 Spark SQL — Menjalankan Kueri SQL di Atas DataFrame

Bagi yang sudah familiar dengan SQL, Spark memungkinkan kita menuliskan kueri SQL **langsung** terhadap DataFrame, tanpa perlu mengingat sintaks DataFrame API. Kedua pendekatan (DataFrame API dan Spark SQL) **sama-sama dieksekusi oleh mesin optimisasi yang sama** di balik layar — jadi tidak ada penalti performa, murni soal preferensi gaya penulisan kode.

**Langkah 1: Daftarkan DataFrame sebagai "tabel sementara" (*temporary view*)**

In [7]:
df_transaksi.createOrReplaceTempView("transaksi")
df_produk.createOrReplaceTempView("produk")

print("Kedua tabel sementara berhasil didaftarkan: 'transaksi' dan 'produk'")

Kedua tabel sementara berhasil didaftarkan: 'transaksi' dan 'produk'


**Langkah 2: Jalankan kueri SQL menggunakan `spark.sql()`**

In [8]:
hasil_sql = spark.sql('''
    SELECT t.kategori, p.manager, SUM(t.pendapatan) AS total_pendapatan
    FROM transaksi t
    JOIN produk p ON t.kategori = p.kategori
    GROUP BY t.kategori, p.manager
    ORDER BY total_pendapatan DESC
''')
hasil_sql.show()

+--------------------+-------+----------------+
|            kategori|manager|total_pendapatan|
+--------------------+-------+----------------+
|Kesehatan & Kecan...|   Dewi|        17607693|
|        Rumah Tangga|    Eka|        17243718|
|             Fashion|   Budi|        16588742|
|          Elektronik|   Andi|        16300808|
|   Makanan & Minuman|  Citra|        13338394|
+--------------------+-------+----------------+



Perhatikan bahwa hasil kueri SQL di atas **identik** dengan hasil `join()` + `groupBy()` yang kita tulis dengan DataFrame API pada Sub-bab 5.1 — hanya berbeda gaya penulisan.

### Kapan Memakai DataFrame API, Kapan Memakai Spark SQL?

| Situasi | Rekomendasi |
|---|---|
| Kueri sederhana, tim terbiasa SQL, atau berasal dari sistem lain yang sudah pakai SQL | **Spark SQL** |
| Transformasi kompleks bertahap, butuh reusable function, integrasi erat dengan kode Python lain | **DataFrame API** |
| Ingin fleksibilitas mencampur logika pemrograman (percabangan, perulangan) dengan transformasi data | **DataFrame API** |

Pada praktiknya, banyak data engineer **mencampur keduanya** sesuai kebutuhan — persis seperti yang akan anda lakukan pada Tugas Mandiri minggu ini.

---

## Menutup SparkSession

In [ ]:
spark.stop()
print("SparkSession ditutup.")

---
## Latihan Mandiri

Jalankan ulang cell pembuatan `SparkSession` dan kedua DataFrame (`df_transaksi`, `df_produk`) dari awal modul sebelum mengerjakan latihan berikut.

In [9]:
# Persiapan ulang untuk latihan
spark = SparkSession.builder.appName("Latihan5").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

np.random.seed(7)
df_produk = spark.createDataFrame(pd.DataFrame(data_produk))
df_transaksi = spark.createDataFrame(pd.DataFrame(data_transaksi))
df_transaksi.createOrReplaceTempView("transaksi")
df_produk.createOrReplaceTempView("produk")
print("Siap untuk latihan.")

Siap untuk latihan.


**Soal 1.** Menggunakan DataFrame API (`join`), tampilkan seluruh transaksi kota `"Solo"` beserta nama `manager` dari kategorinya masing-masing.

In [10]:
# Jawaban Soal 1 di sini
df_gabung = df_transaksi.filter(col("kota") == "Solo") \
    .join(df_produk, on="kategori", how="left") \
    .select("order_id", "kategori", "kota", "pendapatan", "manager")
df_gabung.show()

+--------+--------------------+----+----------+-------+
|order_id|            kategori|kota|pendapatan|manager|
+--------+--------------------+----+----------+-------+
|      O2|Kesehatan & Kecan...|Solo|    452308|   Dewi|
|     O15|Kesehatan & Kecan...|Solo|    435461|   Dewi|
|     O10|          Elektronik|Solo|     72294|   Andi|
|     O12|          Elektronik|Solo|     75566|   Andi|
|     O14|          Elektronik|Solo|    459719|   Andi|
|     O21|          Elektronik|Solo|    176776|   Andi|
|     O22|          Elektronik|Solo|     50323|   Andi|
|     O24|          Elektronik|Solo|    349635|   Andi|
|      O7|             Fashion|Solo|    108370|   Budi|
|     O19|        Rumah Tangga|Solo|    137176|    Eka|
|     O32|          Elektronik|Solo|    464460|   Andi|
|     O44|          Elektronik|Solo|    232991|   Andi|
|     O46|          Elektronik|Solo|    226277|   Andi|
|     O49|             Fashion|Solo|    149756|   Budi|
|     O39|        Rumah Tangga|Solo|    376424| 

**Soal 2.** Menggunakan **window function**, tentukan transaksi dengan `pendapatan` **tertinggi** (peringkat 1 saja) di **setiap kota** (bukan kategori). tidak boleh menggunakan `rank()`.

In [11]:
# Jawaban Soal 2 di sini
window_kota = Window.partitionBy("kota").orderBy(col("pendapatan").desc())

df_peringkat = df_transaksi.withColumn("peringkat", row_number().over(window_kota)) \
    .filter(col("peringkat") == 1) \
    .orderBy("kota")

df_peringkat.show()

+--------+------------+--------+----------+---------+
|order_id|    kategori|    kota|pendapatan|peringkat|
+--------+------------+--------+----------+---------+
|     O96|     Fashion|Magelang|    498770|        1|
|    O147|Rumah Tangga|Semarang|    497370|        1|
|    O286|Rumah Tangga|    Solo|    497826|        1|
+--------+------------+--------+----------+---------+



**Soal 3.** Menggunakan **Spark SQL** (bukan DataFrame API), tulis kueri untuk menghitung rata-rata `pendapatan` per `kota`, urutkan dari tertinggi.

In [12]:
# Jawaban Soal 3 di sini
pendapatan_rata_rata = spark.sql("""
    SELECT kota, AVG(pendapatan) AS rata_rata_pendapatan
    FROM transaksi
    GROUP BY kota
    ORDER BY rata_rata_pendapatan DESC
""")

pendapatan_rata_rata.show()

+--------+--------------------+
|    kota|rata_rata_pendapatan|
+--------+--------------------+
|    Solo|  278926.25925925927|
|Semarang|            267303.8|
|Magelang|   263705.6568627451|
+--------+--------------------+



Soal 4 (Refleksi singkat).** Dalam 2-3 kalimat: menurut anda, dalam situasi seperti apa anda akan lebih memilih menulis Spark SQL dibanding DataFrame API pada pekerjaan anda nanti? Tulis jawaban pada markdown cell di bawah ini.

*(Tulis jawaban di sini)*
Saya memilih menggunakan spark SQL ketika melakukan kueri sederhana seperti SELECT,JOIN,GROUP BY dan ORDER BY. Saya akan memilih menggunakan dataFrame API ketika membutuhkan transformasi data yang kompleks  dab bertahap, saat terdapat logika python seprti percabangan atau perulangan.Untuk ukuran data tidak menentukan karena keduanya dieksekusi oleh optimizer sama sehingga setara.

---
## TUGAS MANDIRI (Dikerjakan Selama 1 Minggu)

> **Tenggat waktu:** dikumpulkan paling lambat **sebelum Pertemuan 6 dimulai**.
> **Sifat tugas:** individu.

### Konteks / Skenario

Manajemen platform e-commerce meminta dibuatkan **dashboard performa cabang toko** yang menggabungkan data transaksi (yang sudah ada di HDFS sejak Pertemuan 3-4) dengan data referensi target penjualan tiap cabang. Anda ditugaskan menyiapkan analisis ini menggunakan kombinasi **join, window function, dan Spark SQL** — persis seperti yang dipelajari hari ini.

### Menyiapkan Dataset

Jalankan cell berikut untuk membuat **dua tabel** dan mengunggah tabel transaksi ke HDFS (tabel target cukup dibuat langsung sebagai Spark DataFrame, karena berukuran kecil dan jarang berubah — praktik umum untuk tabel referensi/*dimension table*).

In [13]:
import numpy as np
import pandas as pd

# Tabel 1: Target & PIC per cabang kota (tabel referensi, dibuat langsung sebagai DataFrame)
data_target_cabang = {
    "kota": ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"],
    "target_bulanan": [45000000, 60000000, 55000000, 40000000, 30000000],
    "pic_cabang": ["Rani", "Joko", "Sari", "Bayu", "Fitri"],
}

# Tabel 2: Data transaksi (disimpan sebagai CSV, lalu diunggah ke HDFS)
np.random.seed(55)
n = 500
kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga"]
kota_list = ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"]
data_transaksi_t5 = {
    "order_id": [f"TRX-{i}" for i in range(n)],
    "kategori": np.random.choice(kategori_list, size=n),
    "kota": np.random.choice(kota_list, size=n),
    "unit_terjual": np.random.randint(1, 10, size=n),
    "harga_satuan": np.random.choice([25000, 50000, 75000, 100000, 150000], size=n),
}
pd.DataFrame(data_transaksi_t5).to_csv("transaksi_tugas5.csv", index=False)

!hdfs dfs -mkdir -p /user/irkham/tugas5
!hdfs dfs -put -f transaksi_tugas5.csv /user/irkham/tugas5/
print("Dataset siap. Tabel transaksi sudah diunggah ke HDFS: /user/irkham/tugas5/transaksi_tugas5.csv")
print("Simpan juga 'data_target_cabang' di atas — kalian akan membuatnya menjadi DataFrame sendiri di notebook tugas.")

Dataset siap. Tabel transaksi sudah diunggah ke HDFS: /user/irkham/tugas5/transaksi_tugas5.csv
Simpan juga 'data_target_cabang' di atas — kalian akan membuatnya menjadi DataFrame sendiri di notebook tugas.


### Instruksi Pengerjaan

Buat notebook baru **`Tugas5_[NPM]_[Nama Lengkap].ipynb`**, buat `SparkSession`, lalu:
1. Baca `transaksi_tugas5.csv` **dari HDFS** menjadi `df_transaksi`, tambahkan kolom `pendapatan` (`unit_terjual x harga_satuan`).
2. Buat `df_target` dari dictionary `data_target_cabang` di atas

Kerjakan bagian **A sampai D** berikut:

---

**A. Join & Perbandingan Target** *(bobot 25%)*

Ringkas total `pendapatan` per `kota` dari `df_transaksi`, lalu **join** dengan `df_target`. Tambahkan kolom `pencapaian_persen`. Urutkan hasil dari pencapaian tertinggi.

**B. Window Function — Kategori Terlaris per Kota** *(bobot 25%)*

Menggunakan window function, tentukan **kategori dengan pendapatan tertinggi di setiap kota** (top-1 saja, gunakan `row_number()`).

**C. Spark SQL** *(bobot 25%)*

Daftarkan `df_transaksi` dan `df_target` sebagai *temporary view*, lalu **tulis satu kueri SQL** (bukan DataFrame API) yang menampilkan: `kota`, `pic_cabang`, dan jumlah transaksi (`COUNT`) di kota tersebut, diurutkan dari jumlah transaksi terbanyak.

**D. Kesimpulan** *(bobot 25%)*

Tulis pada markdown cell (**minimal 100 kata**): berdasarkan hasil bagian A dan B, **cabang mana yang berkinerja paling baik** dan **cabang mana yang paling perlu perhatian manajemen**? Sertakan angka-angka pendukung dari hasil analisis kalian, bukan opini tanpa dasar data.

---

### Ketentuan Pengumpulan

- Kumpulkan `Tugas5_[NPM]_[Nama Lengkap].ipynb` melalui Asisten Praktikum, paling lambat **1 minggu dari hari ini, pukul 23.59 WIB**.
- Pastikan Hadoop aktif dan seluruh cell sudah dijalankan (**Run All**) sebelum dikumpulkan.
- Bagian A & B **wajib** menggunakan DataFrame API; bagian C **wajib** menggunakan Spark SQL murni (`spark.sql(...)`).

### Rubrik Penilaian

| Bagian | Kriteria | Bobot |
|---|---|---|
| A. Join & Pencapaian Target | Join benar, kolom `pencapaian_persen` terhitung tepat | 25% |
| B. Window Function | Kategori terlaris per kota teridentifikasi dengan benar menggunakan `row_number()` | 25% |
| C. Spark SQL | Kueri SQL berjalan benar & menghasilkan output yang sesuai instruksi | 25% |
| D. Kesimpulan | Analisis berbasis data, jelas menyebutkan cabang terbaik & yang perlu perhatian | 25% |
